Zadanie zaczynamy od sparsowania plików z danymi oraz załadowania ich do  Dataframeów z biblioteki Pandas. 
Usuwamy pliki **agency.csv** oraz **feed_info.csv**, jako że nie zawierają potrzebnych informacji

In [6]:
import os
import pandas as pd 
from pathlib import Path
data_path = os.path.join(os.getcwd(), "google_transit")
print(data_path)
entries = os.listdir(data_path)
entries =  [x for x in entries if x not in ["agency.csv", "feed_info.csv"]]
dfs = {}
for entry in entries:
    dfs[entry] = (pd.read_csv(os.path.join(data_path ,entry)))

c:\Users\mikka\Desktop\SEM6\SI\Kurs-Sztuczna-inteligencja-\Lista1\google_transit


Zdefiniujmy klasy którymi będziemy przedstawiać nasz graf: 
1. **Edge** - klasa krawędzi
2. **Node** - klasa wierzchołka
3. **Transit Graph** - klasa grafu, agreguje klasy **Node** oraz **Edge**

Przetwarzamy utworzone wcześniej Dataframy, do postaci zdefiniowanych klas. Importujemy zbudowane klasy i helpery



In [7]:
from data_loader import create_transfer_edges, load_calendar, load_calendar_dates, load_edges, load_stops
from models import TransitCalendar, TransitGraph
from utils import reconstruct_path, res_printing_short, format_time, res_printing, get_stop_name, res_printing_summary
from cost_functions import h_cost_time, h_cost_transfers, g_cost_time, g_cost_transfers

df_stops = dfs["stops.csv"]
df_routes = dfs["routes.csv"]
df_trips = dfs["trips.csv"]
df_stop_times = dfs["stop_times.csv"]
df_calendar  = dfs["calendar.csv"]
df_calendar_dates = dfs["calendar_dates.csv"]

graph = TransitGraph()
nodes, grouped_nodes = load_stops(df_stops)
graph.add_nodes(nodes)

transfer_edges = create_transfer_edges(grouped_nodes, 60)
graph.add_edges(transfer_edges)
load_edges(df_routes, df_trips, df_stop_times, graph)


calendar = TransitCalendar()
calendar.set_exceptions(load_calendar_dates(df_calendar_dates))
calendar.set_schedules(load_calendar(df_calendar))
graph.set_calendar(calendar)

graph.check_content()

Created 1108 nodes
Created 2362 transfer edges
Created 45486 edges
Created 1123 exceptions
Created 3263 schedules
Nodes count 1108
Edges count 47847


Implementacja algorytmu A*

In [8]:
import heapq

def A_star(start_id: str, end_id: str, start_time: int, start_date: str, graph: TransitGraph, h_cost_func, g_cost_func):
    open_set = []
    
    start_g = g_cost_func(current_g=0, is_transfer=False, arrival_time=start_time, is_start=True)
    
    g_scores = {start_id: start_g}
    came_from = {}
    
    start_node = graph.nodes[start_id]
    end_node = graph.nodes[end_id]
    h_start = h_cost_func(start_node, end_node) 
    
    heapq.heappush(open_set, (start_g + h_start, start_g, start_time, start_id))
    
    closed_set = set()

    while len(open_set) > 0:
        curr_f, curr_g, curr_time, curr_id = heapq.heappop(open_set)

        curr_node = graph.nodes[curr_id]
        is_goal = (curr_id == end_id)
        if not is_goal and curr_node.parent_station != "":
            if curr_node.parent_station == end_id:
                is_goal = True
            elif curr_node.parent_station == end_node.parent_station:
                is_goal = True
                
        if is_goal:
            return reconstruct_path(came_from, curr_id)
            
        if curr_id in closed_set:
            continue
            
        closed_set.add(curr_id)
        
        neighbours = graph.get_valid_neighbours(curr_id, curr_time, start_date)
        
        for edge, arrival_time, is_transfer in neighbours:
            next_id = edge.target_stop_id
            
            if next_id in closed_set:
                continue
                
            tentative_g = g_cost_func(
                current_g=curr_g, 
                is_transfer=is_transfer, 
                arrival_time=arrival_time, 
                is_start=False
            )
            
            if next_id not in g_scores or tentative_g < g_scores[next_id]:
                came_from[next_id] = {
                    'prev_node': curr_id,
                    'edge_used': edge,
                    'is_transfer': is_transfer
                }
                
                g_scores[next_id] = tentative_g
                
                next_node_obj = graph.nodes[next_id]
                h_cost = h_cost_func(next_node_obj, end_node)
                
                f_cost = tentative_g + h_cost
                
                heapq.heappush(open_set, (f_cost, tentative_g, arrival_time, next_id))
                
    return None





Implementacja Dijkstry

In [9]:
import sys
def dijkstra(start_id: str, end_id: str, start_time: int, start_date: str, graph: TransitGraph, h_cost_func, g_cost_func): 
    open_set = []
    
    start_g = g_cost_func(current_g=0, is_transfer=False, arrival_time=start_time, is_start=True)
    
    g_scores = {start_id: start_g}
    came_from = {}
    
    start_node = graph.nodes[start_id]
    end_node = graph.nodes[end_id]
    
    heapq.heappush(open_set, (start_g, start_g, start_time, start_id))
    
    closed_set = set()

    while len(open_set) > 0:
        curr_f, curr_g, curr_time, curr_id = heapq.heappop(open_set)

        curr_node = graph.nodes[curr_id]
        is_goal = (curr_id == end_id)
        if not is_goal and curr_node.parent_station != "":
            if curr_node.parent_station == end_id:
                is_goal = True
            elif curr_node.parent_station == end_node.parent_station:
                is_goal = True
                
        if is_goal:
            return reconstruct_path(came_from, curr_id)
            
        if curr_id in closed_set:
            continue
            
        closed_set.add(curr_id)
        
        neighbours = graph.get_valid_neighbours(curr_id, curr_time, start_date)
        if not neighbours:
            print("Brak sąsiadów dla", curr_id, "o czasie", curr_time)
        
        for edge, arrival_time, is_transfer in neighbours:
            next_id = edge.target_stop_id
            
            if next_id in closed_set:
                continue
                
            tentative_g = g_cost_func(
                current_g=curr_g, 
                is_transfer=is_transfer, 
                arrival_time=arrival_time, 
                is_start=False
            )
            
            if next_id not in g_scores or tentative_g < g_scores[next_id]:
                came_from[next_id] = {
                    'prev_node': curr_id,
                    'edge_used': edge,
                    'is_transfer': is_transfer
                }
                
                g_scores[next_id] = tentative_g
                
                # For Dijkstra, f_cost = g_cost (no heuristic)
                f_cost = tentative_g
                
                heapq.heappush(open_set, (f_cost, tentative_g, arrival_time, next_id))
                
    return None 
    


Interfejsy dla zadań

In [10]:
import time

def zad1(station_A: str, station_B: str, criteria: str, start_time: str, start_date: str, algorithm: str):
    
    if station_A not in graph.nodes:
        print(f"BŁĄD: Brak stacji startowej {station_A} w grafie!")
        return []
    if station_B not in graph.nodes:
        print(f"BŁĄD: Brak stacji docelowej {station_B} w grafie!")
        return []
        
    match algorithm.lower(): 
        case 'a*':
            chosen = A_star
        case 'dijkstra': 
            chosen = dijkstra
        case _: 
            print("Wybrano nieprawidłowy algorytm.")
            return

    match criteria.lower(): 
        case 't': 
            h = h_cost_time
            g = g_cost_time
        case 'p': 
            h = h_cost_transfers
            g = g_cost_transfers
        case _:
            print("Wybrano nieprawidłowe kryterium.")
            return 
    alg_start = time.perf_counter()
    wynik = chosen(station_A, station_B, start_time,start_date, graph,h, g)
    alg_end  = time.perf_counter()
    duration  = alg_end - alg_start
    print(f"Czas przeszukiwania trasy: {duration:.4f}s")
    return wynik




# res = zad1('1413380', '1474662', 't', 3660*9 , '20260525', 'a*')
data = '20260525'
timer = 3600*9
tests = [('1413334', '1413380', 't', timer, data, 'a*'), 
         ('1413334', '1413066', 't', timer, data, 'a*'), 
         ('1413066', '1474861', 't', timer, data, 'a*')]



time2 =0
data2 =data
tests_two_metrics2= [('1413064', '2246783', time2, data2),
                    ('1413083', '1413104', time2, data2,),
                    ('1413137', '1474618', time2, data2,)
                    ]
tests_two_metrics = [('1474849', '1474743' , time2, data2 ),
                    ('1536290', '1413427' , time2, data2 ), 
                    ('1413374', '1413328' , time2, data2 )]
print_function = res_printing_summary
# res_printing(res, df_stops)


# for entry in tests:
#     res = zad1(entry[0],entry[1], entry[2], entry[3], entry[4], entry[5])
#     res_printing(res, df_stops)



print("Dijkstra")
for i, entry in enumerate(tests_two_metrics, 0):
    print(f"TEST {get_stop_name(df_stops, entry[0])} - {get_stop_name(df_stops, entry[1])} - czas")
    res = zad1(entry[0],entry[1], 't',entry[2], entry[3], "dijkstra")
    print_function(res, df_stops)
    print(f"TEST {get_stop_name(df_stops, entry[0])} - {get_stop_name(df_stops, entry[1])} - przesiadki")
    res = zad1(entry[0],entry[1], 'p',entry[2], entry[3], "dijkstra")
    print_function(res, df_stops)

print("A*")
for i, entry in enumerate(tests_two_metrics, 0):
    print(f"TEST {get_stop_name(df_stops, entry[0])} - {get_stop_name(df_stops, entry[1])} - czas")
    res = zad1(entry[0],entry[1], 't' ,entry[2], entry[3], "a*")
    print_function(res, df_stops)
    print(f"TEST {get_stop_name(df_stops, entry[0])} - {get_stop_name(df_stops, entry[1])} - przesiadki")
    res = zad1(entry[0],entry[1], 'p',entry[2], entry[3], "a*")
    print_function(res, df_stops)



Dijkstra
TEST Bardo Przyłęk - Trzebieszowice - czas
Brak sąsiadów dla 1475162 o czasie 82740
Brak sąsiadów dla 1474891 o czasie 85980
Brak sąsiadów dla 1475082 o czasie 87540
Czas przeszukiwania trasy: 0.1146s
Brak trasy do wyświetlenia (path jest puste lub None).
TEST Bardo Przyłęk - Trzebieszowice - przesiadki
Brak sąsiadów dla 1475162 o czasie 82740
Brak sąsiadów dla 1474891 o czasie 85980
Brak sąsiadów dla 1475082 o czasie 87540
Czas przeszukiwania trasy: 0.1162s
Brak trasy do wyświetlenia (path jest puste lub None).
TEST Zielona Góra Nowy Kisielin - Żórawina - czas
Czas przeszukiwania trasy: 0.0315s
Trasa: Zielona Góra Nowy Kisielin -> Żórawina | found: True | czas: 176 min | przesiadki: 1 | start: 05:40:00 | koniec: 08:36:00
TEST Zielona Góra Nowy Kisielin - Żórawina - przesiadki
Czas przeszukiwania trasy: 0.0262s
Trasa: Zielona Góra Nowy Kisielin -> Żórawina | found: True | czas: 205 min | przesiadki: 0 | start: 05:11:00 | koniec: 08:36:00
TEST Wierzbowa Śląska - Suszka - czas
C